# 03 — Descubrimiento + scoring en vivo

**Checkpoint: Día 1 — revisión de avance.** Corre el pipeline completo (`saberlink/pipeline.py::run_query`) sobre un ID real y muestra el ranking con su desglose de 4 señales — nunca un score sin explicar.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from saberlink import pipeline

pd.set_option('display.max_colwidth', 60)

## Consulta 1 — NEED-001 (deserción / permanencia estudiantil)

La primera llamada del proceso paga el costo de cargar el modelo de embeddings (~20s). Las siguientes corren en 1-2s — así se ve en la demo real: el proceso queda corriendo y cada consulta nueva es rápida.

In [2]:
out = pipeline.run_query(entity_id='NEED-001', top_k=8)
print(f"tiempo: {out['meta']['elapsed_seconds']}s | candidatos evaluados: {out['meta']['total_candidates_scored']}")

rows = []
for r in out['results']:
    b = r['relevance']['breakdown']
    rows.append({
        'tipo': r['target']['type'], 'id': r['target']['id'],
        'score': r['relevance']['score'], 'label': r['relevance']['label'],
        'semantica': b['semantic'], 'dominio': b['domain'],
        'metodo': b['method'], 'estructural': b['structural'],
    })
pd.DataFrame(rows)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tiempo: 44.828s | candidatos evaluados: 2026


,tipo,id,score,label,semantica,dominio,metodo,estructural
0,THS,THS-012,0.6545,media,0.7535,0.8037,None,0.1250
1,THS,THS-007,0.6472,media,0.7535,0.7844,None,0.1250
2,THS,THS-006,0.6342,media,0.7573,0.7452,None,0.1250
3,PRJ,PRJ-001,0.6299,media,0.7075,0.7353,None,0.2381
4,THS,THS-001,0.6268,media,0.7726,0.7074,None,0.1250
5,THS,THS-016,0.6205,media,0.7573,0.7088,None,0.1250
6,COM,COM-0052,0.5924,media,0.4970,1.0000,None,0.0000
7,COM,COM-0066,0.5924,media,0.4970,1.0000,None,0.0000


## Explicación y evidencia del resultado #1

La explicación es un template de texto que solo inserta números ya calculados en el breakdown — nunca puede inventar una relación que no esté sustentada. La evidencia siempre se relee de `entities.parquet` en el momento, nunca de una copia guardada.

In [3]:
top = out['results'][0]
print(top['explanation'])
print()
for ev in top['evidence']:
    print(f"- {ev['file']} / {ev['id']} / {ev['field']}:")
    print(f"    {ev['snippet']}")

THS-012 es un resultado de relevancia media (score=0.65) para NEED-001. Similitud semántica 0.75 entre el campo 'description' de NEED-001 y 'problem_statement' de THS-012. Comparte los términos de dominio: riesgo académico, trayectorias educativas (dominio=0.80). Señal de metodología no aplica: la fuente no expone un campo de metodología por diseño. Proximidad estructural en el grafo institucional: NEED-001 -[NEED.originating_unit(regex)]-> FAC-004 ; FAC-004 -[PRG.faculty_id]-> PRG-012 ; PRG-012 -[THS.program_id]-> THS-012 (estructural=0.12).

- 03_knowledge_needs/institutional_needs.csv / NEED-001 / description:
    La institución requiere fortalecer su capacidad para abordar predicción y prevención de deserción estudiantil mediante el aprovechamiento articulado de información, conocimiento previo y capacidades existentes. El problema se expresa en términos de permanencia estudiantil, riesgo académico y trayectorias educativas, sin prescribir una solución tecnológica específica.
- 03_

## Por qué `método` aparece como 'no aplica' para consultas NEED

`institutional_needs.csv` no tiene columna de metodología por diseño: una necesidad no debe prescribir una solución técnica. Cuando la fuente es NEED, el peso de la señal de método se redistribuye entre semántica/dominio/estructural — nunca se pone en 0 (eso penalizaría injustamente todo resultado NEED-origen por igual).

In [4]:
b = top['relevance']['breakdown_status']
print(b)

{'semantic': 'available', 'domain': 'available', 'method': 'not_applicable', 'structural': 'available'}


## Consulta 2 — NEED-013 (fraude financiero), para confirmar que no es un caso aislado

In [5]:
out2 = pipeline.run_query(entity_id='NEED-013', top_k=5)
print(f"tiempo: {out2['meta']['elapsed_seconds']}s (ya con el modelo caliente)")
for r in out2['results']:
    print(f"  {r['target']['type']} {r['target']['id']}  score={r['relevance']['score']:.2f}")

tiempo: 2.124s (ya con el modelo caliente)
  PRJ PRJ-097  score=0.67
  THS THS-241  score=0.67
  THS THS-246  score=0.67
  THS THS-251  score=0.65
  PRJ PRJ-102  score=0.63


## Regla de interpretación verificada: 'misma facultad no significa mayor pertinencia'

El peso de proximidad estructural (`w4=0.15`) es deliberadamente bajo. Confirmamos que el resultado #1 no depende solo de estar en el mismo camino del grafo, sino de tener semántica o dominio fuertes por sí mismos.

In [6]:
top1 = out['results'][0]['relevance']['breakdown']
semantic = top1['semantic'] or 0
domain = top1['domain'] or 0
structural = top1['structural'] or 0
print(f'semantica={semantic:.2f} dominio={domain:.2f} estructural={structural:.2f}')
assert semantic >= 0.5 or domain >= 0.5, 'el resultado top no debería depender solo de estructura'
print('OK: el resultado top está sustentado en contenido, no solo en cercanía estructural.')

semantica=0.75 dominio=0.80 estructural=0.12
OK: el resultado top está sustentado en contenido, no solo en cercanía estructural.
